In [ ]:
from docx import Document

TEMPLATE_PATH = "LuxuryEvermore New Receipt Template Auto.docx"   # 你的原始模板（带格式）
OUTPUT_PATH   = "receipt_filled2.docx"     # 输出的新文件


def replace_placeholder_in_paragraph(paragraph, placeholder: str, replacement: str) -> None:
    """
    替换一个段落中的 placeholder（支持 placeholder 跨多个 runs）。
    尽量保留原排版；被替换的那一段会使用 placeholder 起始 run 的格式。
    """
    runs = paragraph.runs
    if not runs:
        return

    # 拼接全文 & 记录每个 run 在全文中的区间
    full = ""
    run_spans = []  # (start, end)
    for r in runs:
        start = len(full)
        full += r.text or ""
        end = len(full)
        run_spans.append((start, end))

    if placeholder not in full:
        return

    # 多次出现也要循环替换
    while True:
        idx = full.find(placeholder)
        if idx == -1:
            break

        start_idx = idx
        end_idx = idx + len(placeholder)

        # 找到覆盖的 runs
        start_run = None
        end_run = None
        for i, (s, e) in enumerate(run_spans):
            if start_run is None and s <= start_idx < e:
                start_run = i
            if s < end_idx <= e:
                end_run = i
                break

        if start_run is None or end_run is None:
            break

        # 计算在起止 run 内的偏移
        s_run_start, s_run_end = run_spans[start_run]
        e_run_start, e_run_end = run_spans[end_run]
        s_off = start_idx - s_run_start
        e_off = end_idx - e_run_start

        # 取出起止 run 的原文本
        s_text = runs[start_run].text or ""
        e_text = runs[end_run].text or ""

        # 起始 run：保留前缀 + replacement + (如果 start_run==end_run 则保留后缀)
        prefix = s_text[:s_off]
        if start_run == end_run:
            suffix = s_text[e_off:]
            runs[start_run].text = prefix + str(replacement) + suffix
        else:
            runs[start_run].text = prefix + str(replacement)

            # 中间 runs 清空（占位符覆盖范围内的内容）
            for k in range(start_run + 1, end_run):
                runs[k].text = ""

            # 结束 run：保留后缀
            suffix = e_text[e_off:]
            runs[end_run].text = suffix

        # 替换后需要重新计算 full 和 spans（因为长度变了）
        full = ""
        run_spans = []
        for r in runs:
            st = len(full)
            full += r.text or ""
            en = len(full)
            run_spans.append((st, en))


def replace_in_paragraph_all(paragraph, mapping: dict) -> None:
    # 逐个占位符替换（支持跨 runs）
    for k, v in mapping.items():
        replace_placeholder_in_paragraph(paragraph, k, str(v))


def replace_in_table(table, mapping: dict) -> None:
    for row in table.rows:
        for cell in row.cells:
            for p in cell.paragraphs:
                replace_in_paragraph_all(p, mapping)
            for t in cell.tables:
                replace_in_table(t, mapping)


def replace_everywhere(doc: Document, mapping: dict) -> None:
    # 正文段落
    for p in doc.paragraphs:
        replace_in_paragraph_all(p, mapping)

    # 正文表格
    for table in doc.tables:
        replace_in_table(table, mapping)

    # 页眉页脚（每个 section）
    for section in doc.sections:
        header = section.header
        footer = section.footer

        for p in header.paragraphs:
            replace_in_paragraph_all(p, mapping)
        for table in header.tables:
            replace_in_table(table, mapping)

        for p in footer.paragraphs:
            replace_in_paragraph_all(p, mapping)
        for table in footer.tables:
            replace_in_table(table, mapping)


def main():
    doc = Document(TEMPLATE_PATH)

    fake_data = {
        "{{Buyer}}": "Alice Tan",
        "{{Receipt_no}}": "LE-2025-000128",
        "{{Date}}": "2025-12-23",
        "{{Payment_method}}": "PayNow",
        "{{Payment_status}}": "Paid",
        "{{Additional_notes}}": "N.A.",
        "{{Item_name}}": "Chanel Classic Flap Small (Black Lambskin)",
        "{{Inclusions}}": "Full set: Box, Dustbag, Card",
        "{{Amount}}": "9800",
        "{{Currency}}": "SGD",
        "{{Total_amount}}": "9800",
        "{{Image}}": "[Image Placeholder]",
    }

    replace_everywhere(doc, fake_data)
    doc.save(OUTPUT_PATH)
    print(f"Saved: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()


Saved: receipt_filled2.docx
